In [124]:
import pandas as pd
import numpy as np

In [125]:
df = pd.read_csv(
    "../Data/iris.csv",
    header=None
    )

In [126]:
df.shape

(150, 5)

In [127]:
df.head

<bound method NDFrame.head of        0    1    2    3               4
0    5.1  3.5  1.4  0.2     Iris-setosa
1    4.9  3.0  1.4  0.2     Iris-setosa
2    4.7  3.2  1.3  0.2     Iris-setosa
3    4.6  3.1  1.5  0.2     Iris-setosa
4    5.0  3.6  1.4  0.2     Iris-setosa
..   ...  ...  ...  ...             ...
145  6.7  3.0  5.2  2.3  Iris-virginica
146  6.3  2.5  5.0  1.9  Iris-virginica
147  6.5  3.0  5.2  2.0  Iris-virginica
148  6.2  3.4  5.4  2.3  Iris-virginica
149  5.9  3.0  5.1  1.8  Iris-virginica

[150 rows x 5 columns]>

In [128]:
X_test = df.iloc[:,0:4].sample(frac=0.2, random_state=42)
y_test = df.iloc[:,4:5].sample(frac=0.2, random_state=42)

X_train = df.iloc[:,0:4].drop(X_test.index)
y_train = df.iloc[:,4:5].drop(y_test.index)

X_test = X_test.to_numpy()
y_test = y_test.to_numpy()

X_train = X_train.to_numpy()
y_train = y_train.to_numpy()

In [129]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean", weights="uniform"):
        self.k = k
        self.distance_metric = distance_metric
        self.weights = weights

    def fit(self, X, y):
        """
        Store the 'prior knowledge' of the model, that will be used
        to predict new labels.
        """
        self.stored_X = X
        self.stored_y = y



    def predict(self, X):
        """
        Run the KNN classification on X
        """

        y = []

        for p in X:
            if self.distance_metric == "euclidean":
                dist = self.__euclideanDist(p, self.stored_X)
            elif self.distance_metric == "cosine":
                dist = self.__cosineDist(p, self.stored_X)
            elif self.distance_metric == "manhattan":
                dist = self.__manhattanDist(p, self.stored_X)

            best_ids = np.argsort(dist)
            best_ids = best_ids[:self.k]
            classes = self.stored_y[best_ids]

            if self.weights == "uniform":
                values, counts = np.unique(classes, return_counts=True)
                most_freq = values[np.argmax(counts)]

            elif self.weights == "distance":
                weights = 1/dist[best_ids]
                values = np.unique(classes)
                tot_weights = np.array([
                    weights[classes == c].sum()
                    for c in values
                ])

                most_freq = values[np.argmax(tot_weights)]
            
            y.append(most_freq)

        return np.array(y)


    def __euclideanDist(self, p, q):
        sq_diff = (p - q) ** 2
        point_sum = np.sum(sq_diff, axis=1)
        ed = np.sqrt(point_sum)

        return ed

    def __cosineDist(self, p, q):
        return 1 - np.abs(self.__cosineSim(p, q))

    def __cosineSim(self, p, q):
        num = np.sum(p * q, axis=1)
        den = np.sqrt(np.sum(p * p)) * np.sqrt(np.sum(q * q, axis=1))

        return num/den

    def __manhattanDist(self, p, q):
        return np.sum(np.abs(p - q), axis=1)

In [130]:
knn = KNearestNeighbors(k=5)

knn.fit(X_train, y_train)

In [131]:
y_pred = knn.predict(X_test)


In [132]:
acc = np.mean(y_pred == y_test)
acc

np.float64(0.33555555555555555)

In [133]:
df2 = pd.read_csv("../Data/mnist_test.csv", header=None)

In [134]:
df2.describe()

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
count,10000.000000,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,...,10000.000000,10000.000000,10000.000000,10000.0000,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0
mean,4.443400,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.179300,0.163600,0.052600,0.0006,0.0,0.0,0.0,0.0,0.0,0.0
std,2.895865,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.674149,5.736072,2.420004,0.0600,0.0,0.0,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,253.000000,253.000000,156.000000,6.0000,0.0,0.0,0.0,0.0,0.0,0.0


In [135]:
df2_sample = df2.groupby(0, group_keys=False).sample(n=100, random_state=42)

In [136]:
df2_sample.shape

(1000, 785)

In [137]:
X2_test = df2_sample.iloc[:,1:].sample(frac=0.2, random_state=42)
y2_test = df2_sample.iloc[:,0].sample(frac=0.2, random_state=42)

X2_train = df2_sample.iloc[:,1:].drop(X2_test.index)
y2_train = df2_sample.iloc[:,0].drop(X2_test.index)

X2_test = X2_test.to_numpy()
y2_test = y2_test.to_numpy()

X2_train = X2_train.to_numpy()
y2_train = y2_train.to_numpy()

In [138]:
knn.fit(X2_train, y2_train)

In [139]:
y2_pred = knn.predict(X2_test)


In [140]:
acc = np.mean(y2_pred == y2_test)
acc

np.float64(0.91)

In [141]:

# Validate KNN on MNIST using my own implementation

results = []

for metric in ["euclidean", "cosine", "manhattan"]:
    for weights in ["uniform", "distance"]:
        for k in [2, 5, 10, 20]:

            knn = KNearestNeighbors(
                k=k,
                distance_metric=metric,
                weights=weights
            )

            knn.fit(X2_train, y2_train)
            y2_pred = knn.predict(X2_test)

            acc = np.count_nonzero(y2_pred == y2_test) / len(y2_test)

            results.append({
                "Distance metric": metric,
                "Weights": weights,
                "K-val": k,
                "Accuracy": acc
            })

results_df = pd.DataFrame(results)
display(results_df.sort_values("Accuracy", ascending=False))


,Distance metric,Weights,K-val,Accuracy
12,cosine,distance,2,0.920
4,euclidean,distance,2,0.920
5,euclidean,distance,5,0.920
13,cosine,distance,5,0.920
6,euclidean,distance,10,0.915
8,cosine,uniform,2,0.915
20,manhattan,distance,2,0.910
1,euclidean,uniform,5,0.910
14,cosine,distance,10,0.910
2,euclidean,uniform,10,0.905


In [142]:
df3 = pd.read_csv("../Data/AmesHousing.csv", index_col="Order")

In [143]:
df3.describe()

,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
count,2.930000e+03,2930.000000,2440.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2907.000000,2929.000000,...,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000
mean,7.144645e+08,57.387372,69.224590,10147.921843,6.094881,5.563140,1971.356314,1984.266553,101.896801,442.629566,...,93.751877,47.533447,23.011604,2.592491,16.002048,2.243345,50.635154,6.216041,2007.790444,180796.060068
std,1.887308e+08,42.638025,23.365335,7880.017759,1.411026,1.111537,30.245361,20.860286,179.112611,455.590839,...,126.361562,67.483400,64.139059,25.141331,56.087370,35.597181,566.344288,2.714492,1.316613,79886.692357
min,5.263011e+08,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,12789.000000
25%,5.284770e+08,20.000000,58.000000,7440.250000,5.000000,5.000000,1954.000000,1965.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000,129500.000000
50%,5.354536e+08,50.000000,68.000000,9436.500000,6.000000,5.000000,1973.000000,1993.000000,0.000000,370.000000,...,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,160000.000000
75%,9.071811e+08,70.000000,80.000000,11555.250000,7.000000,6.000000,2001.000000,2004.000000,164.000000,734.000000,...,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,213500.000000
max,1.007100e+09,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,1424.000000,742.000000,1012.000000,508.000000,576.000000,800.000000,17000.000000,12.000000,2010.000000,755000.000000


In [144]:
y3 = df3["House Style"]
X3 = df3.select_dtypes(include=np.number)
X3 = X3.fillna(value=X3.mean())

In [145]:
X3_train = X3.sample(frac=0.8, random_state=42)
y3_train = y3.loc[X3_train.index]

X3_test = X3.drop(X3_train.index)
y3_test = y3.drop(X3_train.index)

X3_test = X3_test.to_numpy()
y3_test = y3_test.to_numpy()

X3_train = X3_train.to_numpy()
y3_train = y3_train.to_numpy()



In [146]:
# Validate KNN on MNIST using my own implementation

results = []

for metric in ["euclidean", "cosine", "manhattan"]:
    for weights in ["uniform", "distance"]:
        for k in [2, 5, 10, 20]:

            knn = KNearestNeighbors(
                k=k,
                distance_metric=metric,
                weights=weights
            )

            knn.fit(X3_train, y3_train)
            y3_pred = knn.predict(X3_test)

            acc = np.count_nonzero(y3_pred == y3_test) / len(y3_test)

            results.append({
                "Distance metric": metric,
                "Weights": weights,
                "K-val": k,
                "Accuracy": acc
            })

results_df = pd.DataFrame(results)
display(results_df.sort_values("Accuracy", ascending=False))


,Distance metric,Weights,K-val,Accuracy
23,manhattan,distance,20,0.639932
15,cosine,distance,20,0.636519
14,cosine,distance,10,0.634812
22,manhattan,distance,10,0.627986
20,manhattan,distance,2,0.622867
21,manhattan,distance,5,0.619454
13,cosine,distance,5,0.614334
17,manhattan,uniform,5,0.612628
12,cosine,distance,2,0.610922
7,euclidean,distance,20,0.607509


In [147]:
mu    = X3_train.mean(axis=0)     # una media per colonna, calcolata SOLO sul train
sigma = X3_train.std(axis=0)
sigma[sigma == 0] = 1

X3_train = (X3_train - mu) / sigma
X3_test  = (X3_test  - mu) / sigma   # stessi mu e sigma del train


In [148]:
# Validate KNN on MNIST using my own implementation

results = []

for metric in ["euclidean", "cosine", "manhattan"]:
    for weights in ["uniform", "distance"]:
        for k in [2, 5, 10, 20]:

            knn = KNearestNeighbors(
                k=k,
                distance_metric=metric,
                weights=weights
            )

            knn.fit(X3_train, y3_train)
            y3_pred = knn.predict(X3_test)

            acc = np.count_nonzero(y3_pred == y3_test) / len(y3_test)

            results.append({
                "Distance metric": metric,
                "Weights": weights,
                "K-val": k,
                "Accuracy": acc
            })

results_df = pd.DataFrame(results)
display(results_df.sort_values("Accuracy", ascending=False))


/var/folders/31/dy_p77hd38b2szb8fxvx7pfh0000gn/T/ipykernel_86635/3895504726.py:41: RuntimeWarning: divide by zero encountered in divide
  weights = 1/dist[best_ids]


,Distance metric,Weights,K-val,Accuracy
21,manhattan,distance,5,0.860068
17,manhattan,uniform,5,0.858362
23,manhattan,distance,20,0.854949
22,manhattan,distance,10,0.854949
18,manhattan,uniform,10,0.853242
19,manhattan,uniform,20,0.841297
20,manhattan,distance,2,0.837884
6,euclidean,distance,10,0.831058
16,manhattan,uniform,2,0.831058
5,euclidean,distance,5,0.829352


In [ ]:
res = []

for k in [2, 5, 10, 20]:
    for i in range(20):
        X_train_var = X3.sample(frac=0.8)
        X_test_var  = X3.drop(X_train_var.index)
        y_train_var = y3.loc[X_train_var.index].to_numpy()
        y_test_var  = y3.loc[X_test_var.index].to_numpy()

        X_train_var = X_train_var.to_numpy()
        X_test_var  = X_test_var.to_numpy()

        mu    = X_train_var.mean(axis=0)
        sigma = X_train_var.std(axis=0)
        sigma[sigma == 0] = 1
        X_train_var = (X_train_var - mu) / sigma
        X_test_var  = (X_test_var  - mu) / sigma

        knn = KNearestNeighbors(k, distance_metric="manhattan")
        knn.fit(X_train_var, y_train_var)
        y_pred_var = knn.predict(X_test_var)

        acc = np.count_nonzero(y_pred_var == y_test_var) / len(y_test_var)
        res.append({"k": k, "iteration": i, "acc": acc})

res = pd.DataFrame(res)
display(res.groupby("k")["acc"].agg(["mean", "std"]))


,mean,std
k,,
2,0.855290,0.014529
5,0.877133,0.012343
10,0.872952,0.011312
20,0.865956,0.014840
